# Decomposition

## Global Importation

In [1]:
# Shape modules 
from collections import defaultdict

# Computational modules 
%matplotlib inline
import xarray as xr
import glob
import os
import numpy as np
import netCDF4
from netCDF4 import Dataset
import pandas as pd
import re
from array import array
from pylab import *
from scipy.stats import ttest_ind

#import geopandas
#from eofs.xarray import Eof
#from eofs.multivariate.standard import MultivariateEof
import random
import cftime

# Plotting modules 
import matplotlib.pyplot as plt

#from mpl_toolkits.basemap import Basemap
import pandas.plotting
import matplotlib.ticker as ticker

#import seaborn as sns
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.colors import BoundaryNorm
from cartopy.util import add_cyclic_point
from shapely.geometry import Polygon as ShapelyPolygon

"""
# Scikit-learn
from sklearn import linear_model
from sklearn.linear_model import LassoCV, LassoLarsCV, LassoLarsIC
from sklearn.linear_model import Lasso
from sklearn.metrics import r2_score
from sklearn import preprocessing
from sklearn import tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
from sklearn import metrics
from sklearn.neural_network import MLPRegressor
from scipy.optimize import minimize
from scipy.optimize import dual_annealing
from sklearn.decomposition import PCA
from scipy.spatial.distance import pdist
import scipy.stats as stats
from scipy.stats import ttest_ind
"""

# Dask
import dask
"""
from dask import delayed, compute
from dask.distributed import Client
"""

# tqdm + Remove warnings
from tqdm import tqdm 
import warnings
warnings.filterwarnings('ignore')

ERROR 1: PROJ: proj_create_from_database: Open of /apps/jasmin/jaspy/miniforge_envs/jaspy3.12/mf3-25.3.0-3/envs/jaspy3.12-mf3-25.3.0-3-v20250704/share/proj failed


## Global Inset

### Load Merged Data (Precipitation ESM-up2p0; Entire Timeline)

In [2]:
path_pr_files = '/home/users/alvin_ch/files/Multi-model/UKESM_ramp_up/Precip/'
complete_timeline_filename = 'ramp_up_r1i1p1f1_time_merged.nc'

ramp_up_1_complete_pr_file = {}

ramp_up_1_complete_pr_file['complete_timeline'] = xr.open_dataset(f'{path_pr_files}{complete_timeline_filename}')

print(f'Start Date: ', ramp_up_1_complete_pr_file['complete_timeline']['time'].min().item())
print(f'End Date: ', ramp_up_1_complete_pr_file['complete_timeline']['time'].max().item())

Start Date:  1850-01-16 00:00:00
End Date:  2139-12-16 00:00:00


In [3]:
ramp_up_1_complete_pr_file['complete_timeline']['pr'] *= 86400

### Load Merged Data (Surface Humidity ESM-up2p0; Entire Timeline)

In [4]:
path_huss_files = '/home/users/alvin_ch/files/Multi-model/UKESM_ramp_up/Humidity/'
complete_timeline_filename = 'time_merge_esm-up2p0_r1i1p1f1.nc'

ramp_up_1_complete_huss_file = {}

ramp_up_1_complete_huss_file['complete_timeline'] = xr.open_dataset(f'{path_huss_files}{complete_timeline_filename}')

print(f'Start Date: ', ramp_up_1_complete_huss_file['complete_timeline']['time'].min().item())
print(f'End Date: ', ramp_up_1_complete_huss_file['complete_timeline']['time'].max().item())

Start Date:  1850-01-16 00:00:00
End Date:  2139-12-16 00:00:00


In [5]:
ramp_up_1_complete_huss_file['complete_timeline']['huss'] *= 100

## Decomposition

### Implementation 1a

In [6]:
# Note that M_eff should not be interpreted as physical convective mass flux.
# It is an effective proxy which represents the processes linking surface moisture to precipitation.

def precipitation_decomposition(precip, humidity, baseline_precip=None, baseline_humidity=None):

    ### BASELINE CLIMATE - CLIMATOLOGY ###
    # If no baseline is provided, use the temporal mean.
    # Compute monthly climatology as the base.
    if baseline_precip is None:
        baseline_precip = precip.groupby("time.month").mean(dim="time")
        
    if baseline_humidity is None:
        baseline_humidity = humidity.groupby("time.month").mean(dim="time")

    # Baseline mass flux is the average of effective mass flux
    baseline_mass_flux = baseline_precip / baseline_humidity

    ### EFFECTIVE CONVECTIVE MASS-FLUX TERM ###
    # Mathematically denoted by M_eff = P / q
    effective_mass_flux = precip / humidity    

    ### DELTA VALUES ###
    # Calculate differences relative to baseline
    delta_precip = precip.groupby("time.month") - baseline_precip
    delta_humidity = humidity.groupby("time.month") - baseline_humidity
    delta_mass_flux = effective_mass_flux.groupby("time.month") - baseline_mass_flux

    # Thermodynamic term
    # ΔP_thermo = M0 Δq
    # i.e. Moisture changes with circulation held fixed.
    thermodynamic = baseline_mass_flux * delta_humidity.groupby("time.month")

    # Dynamic term
    # ΔP_dynamic = q0 ΔM
    # i.e. Circulation changes with moisture held fixed.
    dynamic = baseline_humidity * delta_mass_flux.groupby("time.month")

    # Nonlinear interaction term
    # ΔP_cross = ΔM Δq
    cross_term = delta_mass_flux * delta_humidity

    # Reconstruct total precipitation anomaly
    # This is used to calculate the residual.
    # The residual should be very close to zero.
    # This is because ΔP = ΔP_thermo + ΔP_dynamic + ΔP_cross
    reconstructed_delta_precip = thermodynamic + dynamic + cross_term
    residual = delta_precip - reconstructed_delta_precip

    # Return all components.
    decomposition = xr.Dataset({"effective_mass_flux": effective_mass_flux,
                                "delta_precip": delta_precip,
                                "delta_humidity": delta_humidity,
                                "delta_mass_flux": delta_mass_flux,
                                "thermodynamic": thermodynamic,
                                "dynamic": dynamic,
                                "cross_term": cross_term,
                                "reconstructed_delta_precip": reconstructed_delta_precip,
                                "residual": residual})
    return decomposition.compute()

## Application

In [7]:
# ramp_up_decompose = precipitation_decomposition(ramp_up_1_complete_pr_file['complete_timeline']['pr'],
                                                # ramp_up_1_complete_huss_file['complete_timeline']['huss'])

In [8]:
# Store decomposed data.
# Compute function takes a long time.

path_files = '/home/users/alvin_ch/files/Multi-model/UKESM_ramp_up/Decomposition'
path_figure = '/home/users/alvin_ch/figures/Multi-model/UKESM_ramp_up/Decomposition'

save_names = ['complete_timeline_ramp_up_decompose']

# decomp_results = [ramp_up_decompose]

# for (file, results) in zip(save_names, decomp_results):
 #   ds = results
 #   ds.to_netcdf(f"{path_files}/{file}.nc")

## Further Processing

### Load Decomposition Results

In [9]:
save_names = ['complete_timeline_ramp_up_decompose']

decomposed_raw_results = {}

for file in tqdm(save_names):
    results_open = xr.open_dataset(f"{path_files}/{file}.nc")
    decomposed_raw_results = results_open

100%|███████████████████████████████████████████| 1/1 [00:00<00:00, 31.19it/s]


In [10]:
# Check data quality
decomposed_raw_results['thermodynamic'].values[:1]

array([[[-0.01844637, -0.01834165, -0.01820056, ..., -0.0188739 ,
         -0.0187369 , -0.01851015],
        [-0.01786881, -0.01757421, -0.01777536, ..., -0.01776866,
         -0.01799764, -0.0172492 ],
        [-0.01643896, -0.01740455, -0.01720455, ..., -0.01687193,
         -0.01670046, -0.01594157],
        ...,
        [-0.8071437 , -0.8148639 , -0.8214684 , ..., -0.78690743,
         -0.7919554 , -0.7980827 ],
        [-0.81540823, -0.8195976 , -0.8254562 , ..., -0.79875416,
         -0.8043349 , -0.80996114],
        [-0.7724162 , -0.77083594, -0.7700622 , ..., -0.7738402 ,
         -0.7720975 , -0.77271134]]], shape=(1, 144, 192), dtype=float32)

In [11]:
# Check coordinates and dimensions.
ds = decomposed_raw_results

print(ds.time[0].values)
print(ds.time[-1].values)

print(ds.time.dt.month.values[:20])
print(ds.month.values[:20])

1850-01-16 00:00:00
2139-12-16 00:00:00
[ 1  2  3  4  5  6  7  8  9 10 11 12  1  2  3  4  5  6  7  8]
[ 1  2  3  4  5  6  7  8  9 10 11 12  1  2  3  4  5  6  7  8]


### Standardise Coordinates

In [12]:
def standardise_coordinates(data):
    rename_dict = {}

    if 'lat' in data.coords:
        rename_dict['lat'] = 'latitude'

    if 'lon' in data.coords:
        rename_dict['lon'] = 'longitude'

    if rename_dict:
        data = data.rename(rename_dict)

    # Convert longitude from 0–360 to -180–180
    if data['longitude'].max() > 180:
        data = data.assign_coords(longitude=((data.longitude + 180) % 360) - 180)
        data = data.sortby('longitude')
    return data

### Compute Monthly Average Over Time (ALL MONTHS)

In [45]:
all_months = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

# First 30 years of the ramp-up
start = '1850-01-16'
end = '1881-01-01'

# All start and end dates for each branch-off point.
# Plus or minus 15 years - so each interval is 30 years long.
# Except for 6-degree branch-off window, where we have 20 years, that is +5 and -15.
start_date_list = ["1904-01-01", "1929-01-01", "1977-01-01", "2029-01-01", "2077-01-01", "2109-01-01"]
end_date_list = ["1935-01-01", "1960-01-01", "2008-01-01", "2060-01-01", "2108-01-01", "2140-01-01"]

# 6-degree branch off point does not have sufficient years to do this.
# Hence omitting it.
new_names = ['1.5-degree branch-off', '2-degree branch-off', '3-degree branch-off',
             '4-degree branch-off', '5-degree branch-off', '6-degree branch-off']

anomaly_decomp_ds = {}
first_30yrs_annual_mean = {}
branch_off_window_annual_mean = {}

# Standardise coordinates once
decomp_ds = standardise_coordinates(decomposed_raw_results)

# First 30 years of the ramp-up
first_30yrs_ds = decomp_ds.sel(time=slice(start, end))

# Average over years while keeping each month separate
first_30yrs_monthly = first_30yrs_ds.groupby('time.month').mean(dim='time')

for scenario, start_date, end_date in zip(new_names, start_date_list, end_date_list):

    # Select the 30-year branch-off window
    branch_off_ds = decomp_ds.sel(time=slice(start_date, end_date))

    # Average over years while keeping each month separate
    branch_off_monthly = branch_off_ds.groupby('time.month').mean(dim='time')

    # Mean of the branch-off window minus mean of the first 30 years
    anomaly_decomp_ds[scenario] = branch_off_monthly.mean('month') - first_30yrs_monthly.mean('month')

    # Annual means
    branch_off_window_annual_mean[scenario] = branch_off_ds.groupby('time.year').mean('time')
    first_30yrs_annual_mean[scenario] = first_30yrs_ds.groupby('time.year').mean('time')

In [46]:
# Check dataset dimensions and variables.
anomaly_decomp_ds[new_names[0]]

<xarray.Dataset> Size: 998kB
Dimensions:                     (latitude: 144, longitude: 192)
Coordinates:
  * latitude                    (latitude) float64 1kB -89.38 -88.12 ... 89.38
  * longitude                   (longitude) float64 2kB -179.1 -177.2 ... 179.1
Data variables:
    effective_mass_flux         (latitude, longitude) float32 111kB -1.492 .....
    delta_precip                (latitude, longitude) float32 111kB 0.02698 ....
    delta_humidity              (latitude, longitude) float32 111kB 0.00229 ....
    delta_mass_flux             (latitude, longitude) float32 111kB -1.492 .....
    thermodynamic               (latitude, longitude) float32 111kB 0.02855 ....
    dynamic                     (latitude, longitude) float32 111kB -0.006885...
    cross_term                  (latitude, longitude) float32 111kB 0.005318 ...
    reconstructed_delta_precip  (latitude, longitude) float32 111kB 0.02698 ....
    residual                    (latitude, longitude) float32 111kB -4.254e-1...

In [47]:
# Check data values whether they are the expected range.

for i in new_names:
    print('\n')
    print('Thermodynamic Mean =', anomaly_decomp_ds[i]['thermodynamic'].mean().values)
    print('Dynamic Mean =', anomaly_decomp_ds[i]['dynamic'].mean().values)
    print('Residual Mean =', anomaly_decomp_ds[i]['residual'].mean().values)



Thermodynamic Mean = 0.16544056
Dynamic Mean = -0.1532117
Residual Mean = 2.3349793e-11


Thermodynamic Mean = 0.24314268
Dynamic Mean = -0.21600205
Residual Mean = -7.1653586e-11


Thermodynamic Mean = 0.4073628
Dynamic Mean = -0.33083537
Residual Mean = -1.4140619e-10


Thermodynamic Mean = 0.58393455
Dynamic Mean = -0.4337811
Residual Mean = -7.7961776e-11


Thermodynamic Mean = 0.8185655
Dynamic Mean = -0.55373436
Residual Mean = -8.0461054e-11


Thermodynamic Mean = 0.9276711
Dynamic Mean = -0.6076317
Residual Mean = 8.9585995e-12


In [48]:
print('Delta Precip Mean =', anomaly_decomp_ds[new_names[0]]['delta_precip'].mean().values)

Delta Precip Mean = 0.07173964


In [49]:
lhs = anomaly_decomp_ds[new_names[0]]['delta_precip']
rhs = (anomaly_decomp_ds[new_names[0]]['thermodynamic'] + anomaly_decomp_ds[new_names[0]]['dynamic'] + 
       anomaly_decomp_ds[new_names[0]]['cross_term'] + anomaly_decomp_ds[new_names[0]]['residual'])

print("Minimum closure error:", abs(lhs - rhs).min().item())
print("Mean closure error:", abs(lhs - rhs).mean().item())
print("Maximum closure error:", abs(lhs - rhs).max().item())

Minimum closure error: 0.0
Mean closure error: 3.973973861093327e-08
Maximum closure error: 8.940696716308594e-07


### Compute Monthly Average Over Time (MJJAS)

In [50]:
MJJAS_months = [5, 6, 7, 8, 9]

anomaly_MJJAS_decomp_ds = {}
first_30yrs_MJJAS_mean = {}
branch_off_window_MJJAS_mean = {}

# Standardise coordinates once
decomp_ds = standardise_coordinates(decomposed_raw_results)

# First 50 years of the ramp-up
first_30yrs_ds = decomp_ds.sel(time=slice(start, end))

# Select monsoon months
first_30yrs_MJJAS_ds = first_30yrs_ds.where(first_30yrs_ds.time.dt.month.isin(MJJAS_months), drop=True)

# Average over years while keeping each month separate
first_30yrs_MJJAS_monthly = first_30yrs_MJJAS_ds.groupby('time.month').mean(dim='time')

for scenario, start_date, end_date in zip(new_names, start_date_list, end_date_list):

    # Select the 30-year branch-off window
    branch_off_ds = decomp_ds.sel(time=slice(start_date, end_date))
    branch_off_MJJAS_ds = branch_off_ds.where(branch_off_ds.time.dt.month.isin(MJJAS_months), drop=True)

    # Average over years while keeping each month separate
    branch_off_MJJAS_monthly = branch_off_MJJAS_ds.groupby('time.month').mean(dim='time')

    # Mean of the branch-off window minus mean of the first 30 years
    anomaly_MJJAS_decomp_ds[scenario] = branch_off_MJJAS_monthly.mean('month') - first_30yrs_MJJAS_monthly.mean('month')

    # Annual means
    branch_off_window_MJJAS_mean[scenario] = branch_off_MJJAS_ds.groupby('time.year').mean('time')
    first_30yrs_MJJAS_mean[scenario] = first_30yrs_MJJAS_ds.groupby('time.year').mean('time')

In [51]:
# Check dataset dimensions and variables.
anomaly_MJJAS_decomp_ds[new_names[0]]

<xarray.Dataset> Size: 998kB
Dimensions:                     (latitude: 144, longitude: 192)
Coordinates:
  * latitude                    (latitude) float64 1kB -89.38 -88.12 ... 89.38
  * longitude                   (longitude) float64 2kB -179.1 -177.2 ... 179.1
Data variables:
    effective_mass_flux         (latitude, longitude) float32 111kB -2.904 .....
    delta_precip                (latitude, longitude) float32 111kB 0.02984 ....
    delta_humidity              (latitude, longitude) float32 111kB 0.0007094...
    delta_mass_flux             (latitude, longitude) float32 111kB -2.904 .....
    thermodynamic               (latitude, longitude) float32 111kB 0.03286 ....
    dynamic                     (latitude, longitude) float32 111kB -0.01417 ...
    cross_term                  (latitude, longitude) float32 111kB 0.01115 ....
    reconstructed_delta_precip  (latitude, longitude) float32 111kB 0.02984 ....
    residual                    (latitude, longitude) float32 111kB -3.211e-1...

### Implementation 1b (Student t-Test)

In [52]:
def ttest_NO_bootstrap(data_scen, data_piCTL):
    n_scen = data_scen.shape[0]
    n_ctl = data_piCTL.shape[0]

    assert n_scen == n_ctl
    
    t_stat, p_value = ttest_ind(data_scen, data_piCTL, axis=0, equal_var=False)

    # Create an Xarray
    p_empirical_xr = xr.DataArray(p_value, dims=["latitude", 'longitude'],
                                  coords={"latitude": data_scen["latitude"],
                                          "longitude": data_scen["longitude"]},
                                  name="p_value")

    # Assign a name to your DataArray
    p_empirical_xr.name = "p_value"

    # Convert it to a Dataset (NetCDF requires a Dataset format)
    ds = p_empirical_xr.to_dataset()    
    return ds

### Compute Statistical Significance (ALL MONTHS)

In [53]:
# Dictionary storing p-values for each level.
# This usually takes a while to run.
p_val_dict = {scenario: {} for scenario in new_names}
var_list = ['delta_precip', 'thermodynamic', 'dynamic', 'cross_term']

for scenario in new_names:
    print('='*30)
    print(f'\nStarting {scenario}')
    
    for variable in var_list:
        print(f'  Starting {variable}')

        # Student-t
        ds1 = first_30yrs_annual_mean[scenario][variable]
        ds2 = branch_off_window_annual_mean[scenario][variable]
        %time p_val_dict[scenario][variable] = ttest_NO_bootstrap(ds1, ds2)
        
        print(f'  Finished {variable}\n')


Starting 1.5-degree branch-off
  Starting delta_precip
CPU times: user 17.1 ms, sys: 1.3 ms, total: 18.4 ms
Wall time: 16.2 ms
  Finished delta_precip

  Starting thermodynamic
CPU times: user 15.3 ms, sys: 0 ns, total: 15.3 ms
Wall time: 15.4 ms
  Finished thermodynamic

  Starting dynamic
CPU times: user 13.3 ms, sys: 0 ns, total: 13.3 ms
Wall time: 13.4 ms
  Finished dynamic

  Starting cross_term
CPU times: user 13.6 ms, sys: 0 ns, total: 13.6 ms
Wall time: 13.7 ms
  Finished cross_term


Starting 2-degree branch-off
  Starting delta_precip
CPU times: user 13.6 ms, sys: 0 ns, total: 13.6 ms
Wall time: 13.6 ms
  Finished delta_precip

  Starting thermodynamic
CPU times: user 13.1 ms, sys: 0 ns, total: 13.1 ms
Wall time: 13.1 ms
  Finished thermodynamic

  Starting dynamic
CPU times: user 13.4 ms, sys: 0 ns, total: 13.4 ms
Wall time: 13.5 ms
  Finished dynamic

  Starting cross_term
CPU times: user 13.6 ms, sys: 0 ns, total: 13.6 ms
Wall time: 13.7 ms
  Finished cross_term


Startin

In [55]:
# Check p-values
p_val_dict[new_names[0]]['delta_precip']

<xarray.Dataset> Size: 113kB
Dimensions:    (latitude: 144, longitude: 192)
Coordinates:
  * latitude   (latitude) float64 1kB -89.38 -88.12 -86.88 ... 86.88 88.12 89.38
  * longitude  (longitude) float64 2kB -179.1 -177.2 -175.3 ... 177.2 179.1
Data variables:
    p_value    (latitude, longitude) float32 111kB 1.996e-06 ... 3.75e-08

In [56]:
def save_p_values(p_val_dict, new_names, path_files):

    for scenario in new_names:
        scenario_path = os.path.join(path_files, scenario)
        os.makedirs(scenario_path, exist_ok=True)

        for variable, ds in p_val_dict[scenario].items():
            filename = f'ramp_up_{scenario}_window_{variable}_pvalues.nc'
            filepath = os.path.join(scenario_path, filename)

            ds.to_netcdf(filepath)

In [57]:
save_p_values(p_val_dict, new_names, path_files)

In [58]:
# Check data on saved files.
ds = xr.open_dataset(f'{path_files}/{scenario}/ramp_up_{scenario}_window_{variable}_pvalues.nc')
ds['p_value'].values[:1]

array([[0.55208457, 0.7895752 , 0.7601681 , 0.49127218, 0.6864729 ,
        0.4209017 , 0.39625016, 0.5718671 , 0.31077018, 0.29463106,
        0.30673635, 0.22325738, 0.30502805, 0.18727806, 0.29440263,
        0.2599273 , 0.27983573, 0.31151518, 0.35492408, 0.37841678,
        0.37539065, 0.39124218, 0.40809655, 0.42740652, 0.36132216,
        0.32958624, 0.44662493, 0.4340373 , 0.45006344, 0.506658  ,
        0.5346994 , 0.635463  , 0.6423686 , 0.7021939 , 0.7754109 ,
        0.80942965, 0.8414141 , 0.82886744, 0.8744962 , 0.87430257,
        0.8647448 , 0.84342694, 0.86690545, 0.9130627 , 0.90894365,
        0.93688095, 0.9339406 , 0.97957385, 0.9148647 , 0.9230923 ,
        0.9534815 , 0.9563936 , 0.97187686, 0.98613447, 0.98820764,
        0.967507  , 0.993292  , 0.998917  , 0.98535573, 0.97384894,
        0.9290624 , 0.9059942 , 0.8883428 , 0.8482716 , 0.85504   ,
        0.8189446 , 0.7787402 , 0.70692056, 0.6925166 , 0.6463103 ,
        0.6149756 , 0.59834254, 0.64189434, 0.63

### Compute Statistical Significance (MJJAS)

In [60]:
# Dictionary storing p-values for each level.
# This usually takes a while to run.
p_val_MJJAS_dict = {scenario: {} for scenario in new_names}
var_list = ['delta_precip', 'thermodynamic', 'dynamic', 'cross_term']

for scenario in new_names:
    print('='*30)
    print(f'\nStarting {scenario}')
    
    for variable in var_list:
        print(f'  Starting {variable}')

        # Student-t
        ds1 = first_30yrs_MJJAS_mean[scenario][variable]
        ds2 = branch_off_window_MJJAS_mean[scenario][variable]
        %time p_val_MJJAS_dict[scenario][variable] = ttest_NO_bootstrap(ds1, ds2)
        
        print(f'  Finished {variable}\n')


Starting 1.5-degree branch-off
  Starting delta_precip
CPU times: user 15 ms, sys: 1.36 ms, total: 16.3 ms
Wall time: 15 ms
  Finished delta_precip

  Starting thermodynamic
CPU times: user 14.2 ms, sys: 0 ns, total: 14.2 ms
Wall time: 14.3 ms
  Finished thermodynamic

  Starting dynamic
CPU times: user 13 ms, sys: 0 ns, total: 13 ms
Wall time: 13.1 ms
  Finished dynamic

  Starting cross_term
CPU times: user 13.7 ms, sys: 0 ns, total: 13.7 ms
Wall time: 13.8 ms
  Finished cross_term


Starting 2-degree branch-off
  Starting delta_precip
CPU times: user 13.9 ms, sys: 0 ns, total: 13.9 ms
Wall time: 14 ms
  Finished delta_precip

  Starting thermodynamic
CPU times: user 12.6 ms, sys: 935 μs, total: 13.5 ms
Wall time: 13.6 ms
  Finished thermodynamic

  Starting dynamic
CPU times: user 13.2 ms, sys: 0 ns, total: 13.2 ms
Wall time: 13.3 ms
  Finished dynamic

  Starting cross_term
CPU times: user 13.8 ms, sys: 0 ns, total: 13.8 ms
Wall time: 13.9 ms
  Finished cross_term


Starting 3-deg

In [61]:
# Check p-values
p_val_MJJAS_dict[new_names[0]]['delta_precip']

<xarray.Dataset> Size: 113kB
Dimensions:    (latitude: 144, longitude: 192)
Coordinates:
  * latitude   (latitude) float64 1kB -89.38 -88.12 -86.88 ... 86.88 88.12 89.38
  * longitude  (longitude) float64 2kB -179.1 -177.2 -175.3 ... 177.2 179.1
Data variables:
    p_value    (latitude, longitude) float32 111kB 0.0002285 ... 0.01141

In [62]:
def save_MJJAS_p_values(p_val_dict, new_names, path_files):

    for scenario in new_names:
        scenario_path = os.path.join(path_files, scenario)
        os.makedirs(scenario_path, exist_ok=True)

        for variable, ds in p_val_dict[scenario].items():
            filename = f'ramp_up_{scenario}_window_MJJAS_{variable}_pvalues.nc'
            filepath = os.path.join(scenario_path, filename)

            ds.to_netcdf(filepath)

In [63]:
save_MJJAS_p_values(p_val_MJJAS_dict, new_names, path_files)

In [64]:
# Check data on saved files.
ds = xr.open_dataset(f'{path_files}/{scenario}/ramp_up_{scenario}_window_MJJAS_{variable}_pvalues.nc')
ds['p_value'].values[:1]

array([[0.1188597 , 0.15855446, 0.13855667, 0.09936943, 0.11947494,
        0.08649626, 0.09835512, 0.09877831, 0.06641997, 0.05621593,
        0.06727674, 0.04681518, 0.06381935, 0.03239818, 0.05757492,
        0.0542036 , 0.06559637, 0.07371964, 0.08561551, 0.09797217,
        0.09476984, 0.10820461, 0.12027816, 0.12382092, 0.1080719 ,
        0.10866128, 0.14629097, 0.14898035, 0.15538578, 0.17424366,
        0.17916681, 0.2044571 , 0.21231867, 0.23446812, 0.24972789,
        0.26958892, 0.29084766, 0.2979728 , 0.31183892, 0.32066253,
        0.327492  , 0.3238475 , 0.34194916, 0.3580619 , 0.35229278,
        0.35038504, 0.35040715, 0.3703412 , 0.33119366, 0.3251753 ,
        0.36047894, 0.35874406, 0.36056063, 0.3591678 , 0.36968145,
        0.37238133, 0.3709507 , 0.3641205 , 0.37597743, 0.38855177,
        0.39021522, 0.39647755, 0.4175869 , 0.42456242, 0.42791945,
        0.43279138, 0.45422953, 0.48231015, 0.4960763 , 0.5163962 ,
        0.5402569 , 0.5628613 , 0.54660445, 0.55